# LAnoBERT Kaggle - GitHub từng bước

Notebook clone trực tiếp `https://github.com/rubyhcm/lv-new.git`, tìm thư mục `LAnoBERT`, rồi chạy từng bước trên Kaggle GPU.

In [1]:
from pathlib import Path
import shutil, subprocess, sys, os

GITHUB_URL = 'https://github.com/rubyhcm/lv-new.git'
GITHUB_BRANCH = 'main'
CLONE_DIR = Path('/kaggle/working/lv-new')
PROJECT = Path('/kaggle/working/LAnoBERT')
CONFIG = 'configs/bgl.yaml'
DOWNLOAD_BGL = False
EPOCHS = 1
MAX_EVAL_SAMPLES = 10000
print('GitHub:', GITHUB_URL)
print('Working clone:', CLONE_DIR)


GitHub: https://github.com/rubyhcm/lv-new.git
Working clone: /kaggle/working/lv-new


In [2]:
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Hãy bật Accelerator = GPU trong Kaggle.')


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
os.chdir('/kaggle/working')
if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_URL, str(CLONE_DIR)], check=True)
candidates = [p for p in CLONE_DIR.rglob('lanobert') if p.is_dir() and (p.parent / 'configs').is_dir()]
assert len(candidates) == 1, f'Không tìm thấy duy nhất LAnoBERT: {candidates}'
PROJECT = candidates[0].parent
print('Project:', PROJECT)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip())


Cloning into '/kaggle/working/lv-new'...


Project: /kaggle/working/lv-new/LAnoBERT
Commit: 4d6ed5730b51e3dd51fa75454165c0eda1b3b80c


In [4]:
%pip install -q -r {PROJECT / 'requirements.txt'} pytest

Note: you may need to restart the kernel to use updated packages.


In [5]:
os.chdir(PROJECT)
print('Current directory:', Path.cwd())
print('Config:', Path(CONFIG).exists())
print('BGL directory:', Path('data/BGL').exists())
if Path('data/BGL').exists():
    print(*sorted(str(p) for p in Path('data/BGL').iterdir()), sep='\n')


Current directory: /kaggle/working/lv-new/LAnoBERT
Config: True
BGL directory: False


In [6]:
PROJECT = Path(PROJECT).resolve()
os.chdir(PROJECT)
raw_log = PROJECT / 'data/BGL/BGL.log'
if not raw_log.exists():
    candidates = [p for p in Path('/kaggle/input').rglob('BGL.log') if p.is_file()]
    if candidates:
        source_bgl = candidates[0].parent
        target_bgl = PROJECT / 'data/BGL'
        target_bgl.mkdir(parents=True, exist_ok=True)
        for source in source_bgl.iterdir():
            target = target_bgl / source.name
            if source.is_file() and not target.exists():
                shutil.copy2(source, target)
        print('Copied BGL from:', source_bgl)
    elif DOWNLOAD_BGL:
        subprocess.run(['bash', 'scripts/download_data.sh', 'bgl'], cwd=PROJECT, check=True)
bgl_candidates = list(Path('/kaggle/input').rglob('BGL.log'))
assert raw_log.exists(), f"Thiếu {raw_log}; các file BGL tìm thấy: {bgl_candidates}"
print(raw_log, f'{raw_log.stat().st_size / (1024**3):.2f} GB')


Copied BGL from: /kaggle/input/datasets/jameshcm/dataset-input-0409/BGL
/kaggle/working/lv-new/LAnoBERT/data/BGL/BGL.log 0.69 GB


## Kiểm tra mã nguồn

In [7]:
subprocess.run([sys.executable, '-m', 'pytest', '-v'], check=True)
subprocess.run(['bash', '-n', 'scripts/ensure_data.sh', 'scripts/run_pipeline.sh'], check=True)
print('Tests and shell checks passed')


============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/lv-new/LAnoBERT
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collecting ... collected 7 items

tests/test_field_value.py::test_constraint_engine_emits_auditable_hard_state PASSED [ 14%]
tests/test_field_value.py::test_constraint_engine_does_not_flag_normal_fields PASSED [ 28%]
tests/test_fusion.py::test_gated_fusion_returns_batch_scores_and_normalized_gate PASSED [ 42%]
tests/test_fusion.py::test_freeze_baseline_disables_gradients_and_training_mode PASSED [ 57%]
tests/test_ontology.py::test_ontology_retriever_returns_relevant_document_first PASSED [ 71%]
tests/test_structured_and_early.py::test_extract_fields_supports_bgl_and_key_value_logs PASSED [ 85%]
tests/test_structured_and_early.py::test_early_metrics_measure_warning_before_incident PASSED [100%]

============

## Split và preprocess BGL

In [8]:
subprocess.run(['bash', 'scripts/ensure_data.sh', CONFIG], check=True)

==> [data] waiting for prep lock: data/BGL/.prep.lock
==> [data] acquired lock
    SKIP split (already generated: data/BGL/BGL_train_normal.raw, data/BGL/BGL_test.raw)
    SKIP preprocess train (already generated: data/BGL/BGL_train_normal_parsed.log)
    SKIP preprocess test (already generated: data/BGL/BGL_test_parsed.log)
==> [data] released lock


CompletedProcess(args=['bash', 'scripts/ensure_data.sh', 'configs/bgl.yaml'], returncode=0)

In [9]:
for name in ['BGL_train_normal.raw', 'BGL_test.raw', 'BGL_test_label.log', 'BGL_train_normal_parsed.log', 'BGL_test_parsed.log']:
    path = Path('data/BGL') / name
    print(f'{path}: {path.stat().st_size / (1024**2):.1f} MB' if path.exists() else f'MISSING: {path}')


data/BGL/BGL_train_normal.raw: 489.8 MB
data/BGL/BGL_test.raw: 219.0 MB
data/BGL/BGL_test_label.log: 2.4 MB
data/BGL/BGL_train_normal_parsed.log: 264.4 MB
data/BGL/BGL_test_parsed.log: 134.7 MB


## Train tokenizer

In [10]:
subprocess.run([sys.executable, '-m', 'lanobert.tokenizer', '--config', CONFIG], check=True)
print('Tokenizer ready:', Path('outputs/BGL/tokenizer').exists())





[tokenizer] training on data/BGL/BGL_train_normal_parsed.log
[tokenizer] saved vocab -> outputs/BGL/tokenizer/BGL_LogBERT-vocab.txt
Tokenizer ready: True


## Tạo cấu hình Kaggle

In [11]:
import yaml
with open(CONFIG, encoding='utf-8') as handle:
    runtime = yaml.safe_load(handle)
runtime.setdefault('train', {})['num_train_epochs'] = EPOCHS
runtime.setdefault('inference', {})['max_eval_samples'] = MAX_EVAL_SAMPLES
with open('configs/kaggle_runtime.yaml', 'w', encoding='utf-8') as handle:
    yaml.safe_dump(runtime, handle, sort_keys=False)
print(Path('configs/kaggle_runtime.yaml').read_text())


dataset: BGL
run_name: bgl
paths:
  raw_log: data/BGL/BGL.log
  train_raw: data/BGL/BGL_train_normal.raw
  test_raw: data/BGL/BGL_test.raw
  test_label: data/BGL/BGL_test_label.log
  train_normal: data/BGL/BGL_train_normal_parsed.log
  test_log: data/BGL/BGL_test_parsed.log
  tokenizer_dir: outputs/BGL/tokenizer
  model_dir: outputs/BGL/model
  result_dir: outputs/BGL/results
split:
  method: line
  train_ratio: 0.8
  label_marker: '-'
preprocess:
  regex_profile: bgl
  mask_block_id: true
  mask_ip: true
  mask_number: true
  drop_header_fields: 3
tokenizer:
  vocab_size: 1000
  min_frequency: 2
  lowercase: false
train:
  max_len: 512
  mlm_probability: 0.2
  num_train_epochs: 1
  per_device_train_batch_size: 32
  learning_rate: 0.0001
  weight_decay: 0.01
  warmup_ratio: 0.1
  save_steps: 50000
  save_total_limit: 2
  logging_steps: 1000
  seed: 42
inference:
  hf_model: null
  hf_subfolder: null
  score: error_mean
  top_k: 5
  top_ks:
  - 1
  - 2
  - 3
  - 4
  - 5
  - 6
  - 7
  - 

## Train LAnoBERT baseline

In [12]:
from lanobert.tokenizer import load_tokenizer, vocab_path_for
from lanobert.utils import load_config
from transformers import DataCollatorForLanguageModeling

runtime_cfg = load_config('configs/kaggle_runtime.yaml')
vocab_file = vocab_path_for(runtime_cfg)
train_file = runtime_cfg.get_path('paths.train_normal')
max_train_samples = int(runtime_cfg.get('train.max_train_samples', 10000) or 10000)
tokenizer = load_tokenizer(vocab_file, max_len=512)

print('Project:', Path.cwd())
print('Config:', Path('configs/kaggle_runtime.yaml').resolve())
print('Tokenizer:', Path(vocab_file).resolve())
print('Tokenizer vocab size:', tokenizer.vocab_size)
print('Train corpus:', train_file)
print('Train corpus exists:', Path(train_file).is_file())
print('Train corpus size MB:', Path(train_file).stat().st_size / (1024**2))

assert tokenizer.vocab_size > 5, 'Tokenizer chỉ có special tokens; không được train.'
assert Path(train_file).is_file() and Path(train_file).stat().st_size > 0, 'Training corpus rỗng hoặc thiếu.'

with open(train_file, encoding='utf-8') as handle:
    sample_lines = [line.strip() for _, line in zip(range(8), handle) if line.strip()]
assert sample_lines, 'Không có dòng log hợp lệ trong training corpus.'
encoded = tokenizer(sample_lines, add_special_tokens=False)['input_ids']
content_ids = [token_id for ids in encoded for token_id in ids if token_id not in {
    tokenizer.pad_token_id, tokenizer.unk_token_id, tokenizer.cls_token_id,
    tokenizer.sep_token_id, tokenizer.mask_token_id,
}]
print('Sample line:', sample_lines[0])
print('Sample content token count:', len(content_ids))
assert content_ids, 'Tokenizer không tạo được content token; kiểm tra vocab/corpus.'

from lanobert.dataset import LogLineDataset
dataset = LogLineDataset(tokenizer, train_file, max_len=512, limit=max_train_samples)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.2)
batch = collator([dataset[i] for i in range(min(8, len(dataset)))])
masked_labels = int((batch['labels'] != -100).sum())
print('Dataset examples:', len(dataset))
print('Masked labels in probe batch:', masked_labels)
assert len(dataset) > 0 and masked_labels > 0, 'MLM probe không có label hợp lệ; dừng trước khi train.'
print('Validation OK: có thể bắt đầu training.')


Project: /kaggle/working/lv-new/LAnoBERT
Config: /kaggle/working/lv-new/LAnoBERT/configs/kaggle_runtime.yaml
Tokenizer: /kaggle/working/lv-new/LAnoBERT/outputs/BGL/tokenizer/BGL_LogBERT-vocab.txt
Tokenizer vocab size: 1000
Train corpus: data/BGL/BGL_train_normal_parsed.log
Train corpus exists: True
Train corpus size MB: 264.443058013916
Sample line: server server ip num server ras kernel info instruction cache parity error corrected
Sample content token count: 104
[dataset] pre-tokenizing 10,000 lines...
[dataset] pre-tokenization done.
Dataset examples: 10000
Masked labels in probe batch: 16
Validation OK: có thể bắt đầu training.


In [13]:
subprocess.run([sys.executable, '-m', 'lanobert.train', '--config', 'configs/kaggle_runtime.yaml'], check=True)

[train] loaded tokenizer (vocab=1000)
[train] model params: 86,811,880
[dataset] pre-tokenizing 3,496,193 lines...
[dataset] pre-tokenization done.
[train] examples: train=3,461,232 eval=34,961
[train] transformers does not support TrainingArguments options; skipping: warmup_ratio, logging_dir
[train] start


  0%|          | 0/54082 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
 92%|█████████▏| 50000/54082 [4:32:02<17:09,  3.97it/s]

{'loss': '0.7508', 'grad_norm': '2.91', 'learning_rate': '9.992e-05', 'epoch': '0.01849'}
{'loss': '0.1927', 'grad_norm': '4.732', 'learning_rate': '9.966e-05', 'epoch': '0.03698'}
{'loss': '0.1432', 'grad_norm': '0.7028', 'learning_rate': '9.924e-05', 'epoch': '0.05547'}
{'loss': '0.119', 'grad_norm': '0.6934', 'learning_rate': '9.866e-05', 'epoch': '0.07396'}
{'loss': '0.1042', 'grad_norm': '0.47', 'learning_rate': '9.791e-05', 'epoch': '0.09245'}
{'loss': '0.0895', 'grad_norm': '0.2737', 'learning_rate': '9.699e-05', 'epoch': '0.1109'}
{'loss': '0.09899', 'grad_norm': '0.5253', 'learning_rate': '9.592e-05', 'epoch': '0.1294'}
{'loss': '0.08106', 'grad_norm': '0.7735', 'learning_rate': '9.47e-05', 'epoch': '0.1479'}
{'loss': '0.08658', 'grad_norm': '0.736', 'learning_rate': '9.332e-05', 'epoch': '0.1664'}
{'loss': '0.08', 'grad_norm': '3.728', 'learning_rate': '9.18e-05', 'epoch': '0.1849'}
{'loss': '0.08916', 'grad_norm': '3.103', 'learning_rate': '9.014e-05', 'epoch': '0.2034'}
{'l


100%|█████████▉| 273/274 [01:08<00:00,  3.39it/s]
                                                       
100%|██████████| 274/274 [01:09<00:00,  2.76it/s]
                                                 
Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.03614', 'eval_runtime': '69.97', 'eval_samples_per_second': '499.6', 'eval_steps_per_second': '3.916', 'epoch': '0.9245'}


...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1.78e+04', 'train_samples_per_second': '194.5', 'train_steps_per_second': '3.039', 'train_loss': '0.07208', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.91it/s]


[train] saved final model -> outputs/BGL/model/final


CompletedProcess(args=['/usr/bin/python3', '-m', 'lanobert.train', '--config', 'configs/kaggle_runtime.yaml'], returncode=0)

## Inference và xem kết quả

In [14]:
subprocess.run([sys.executable, '-m', 'lanobert.inference', '--config', 'configs/kaggle_runtime.yaml'], check=True)
result_dir = Path('outputs/BGL/results')
print(*sorted(str(p) for p in result_dir.iterdir()), sep='\n')


[infer] loading trained model from 'outputs/BGL/model/final'


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 2629.26it/s]


[infer] test lines: 10000  labels: 10000
[infer] dedup: 10000 lines -> 1 unique (100.0% saved)


scoring unique: 100%|██████████| 1/1 [00:00<00:00,  2.41it/s]
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1188: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1188: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1188: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/usr/local/lib

[infer] ===== top_k=1 =====
[infer] --- abnormal_error (CE loss), k=1 ---
[eval:BGL_error_k1] AUROC=nan  best_F1=0.0000  thr=2.348e-05
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000   10000.0
           1     0.0000    0.0000    0.0000       0.0

    accuracy                         0.0000   10000.0
   macro avg     0.0000    0.0000    0.0000   10000.0
weighted avg     0.0000    0.0000    0.0000   10000.0

[infer] --- abnormal_prob (top-k largest max_p, original), k=1 ---
[eval:BGL_prob_k1] AUROC=nan  best_F1=0.0000  thr=9.537e-07
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000   10000.0
           1     0.0000    0.0000    0.0000       0.0

    accuracy                         0.0000   10000.0
   macro avg     0.0000    0.0000    0.0000   10000.0
weighted avg     0.0000    0.0000    0.0000   10000.0

[infer] --- abnormal_probtopk (top-k largest 1-max_p), k=1 ---
[eval:BGL_probtopk_k1

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1188: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1188: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1188: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1033: Us


[infer] ===== top_k=7 =====
[infer] --- abnormal_error (CE loss), k=7 ---
[eval:BGL_error_k7] AUROC=nan  best_F1=0.0000  thr=1.264e-05
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000   10000.0
           1     0.0000    0.0000    0.0000       0.0

    accuracy                         0.0000   10000.0
   macro avg     0.0000    0.0000    0.0000   10000.0
weighted avg     0.0000    0.0000    0.0000   10000.0

[infer] --- abnormal_prob (top-k largest max_p, original), k=7 ---
[eval:BGL_prob_k7] AUROC=nan  best_F1=0.0000  thr=1.652e-06
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000   10000.0
           1     0.0000    0.0000    0.0000       0.0

    accuracy                         0.0000   10000.0
   macro avg     0.0000    0.0000    0.0000   10000.0
weighted avg     0.0000    0.0000    0.0000   10000.0

[infer] --- abnormal_probtopk (top-k largest 1-max_p), k=7 ---
[eval:BGL_probtopk_k

In [15]:
for report in sorted(Path('outputs/BGL/results').glob('*report.txt')):
    print(f'\n===== {report.name} =====\n{report.read_text()[:3000]}')
archive = shutil.make_archive('/kaggle/working/lanobert-results', 'zip', 'outputs')
print('Artifact:', archive)



===== BGL_error_k10_report.txt =====
AUROC: nan
best_F1: 0.000000
best_threshold: 9.38168e-06

confusion_matrix:
[[    0 10000]
 [    0     0]]

              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000   10000.0
           1     0.0000    0.0000    0.0000       0.0

    accuracy                         0.0000   10000.0
   macro avg     0.0000    0.0000    0.0000   10000.0
weighted avg     0.0000    0.0000    0.0000   10000.0



===== BGL_error_k1_report.txt =====
AUROC: nan
best_F1: 0.000000
best_threshold: 2.3484e-05

confusion_matrix:
[[    0 10000]
 [    0     0]]

              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000   10000.0
           1     0.0000    0.0000    0.0000       0.0

    accuracy                         0.0000   10000.0
   macro avg     0.0000    0.0000    0.0000   10000.0
weighted avg     0.0000    0.0000    0.0000   10000.0



===== BGL_error_k2_report.txt =====
AUROC: nan
best_F